# 04b · Model selection: accuracy vs. explainability

Chooses the model for the explanation phases (3–6) from **phase 1** (accuracy, notebook 03) and **phase 2**
(heatmap faithfulness, notebook 04). No GPU needed; it only reads saved results and runs in about a minute.

**Notebook settings (right sidebar)**
- **Accelerator:** None · **Internet:** On
- **Inputs:** the committed outputs of **03_evaluate** and **04_gradcam** (both run with *Save & Run All*)

Criteria are **fixed in advance and weighted equally**, and each is ranked across models (1 = best):
| Criterion | Measure | Better |
|---|---|---|
| Accuracy | FF++ test AUC, Celeb-DF test AUC (phase 1) | higher |
| Faithfulness | Grad-CAM++ gain over a random map: deletion (random − Grad-CAM++) and insertion (Grad-CAM++ − random), per dataset | higher |
| Face focus | share of Grad-CAM++ activation inside the face regions | higher |
| Cost | GFLOPs per image | lower |

The overall score is the **mean rank**. The faithfulness chart shows each model's gain with a 95% CI from paired
per-image differences; faded, hatched bars marked *n.s.* are **not** significantly better than random (Wilcoxon p ≥ 0.05).

In [ ]:
# --- Project code -------------------------------------------------------------------------
# Always clones the latest code. It lives in /tmp, OUTSIDE /kaggle/working, so it is never saved into this
# notebook's output (otherwise later notebooks would pick up a stale copy). For a private repo add a
# GITHUB_TOKEN secret. Fallback: a zip of the project uploaded as a Kaggle *dataset*.
import os, sys, glob, shutil, subprocess

PROJECT = "/tmp/dfdc-project"
REPO, BRANCH = "https://github.com/Amay-M-Nair/DFDC.git", "dev"

if not os.path.exists(PROJECT):
    token = ""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT], capture_output=True)
    if r.returncode == 0:
        subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO])  # don't keep the token
    else:
        found = [p for d in range(5) for p in glob.glob("/kaggle/input/" + "*/" * d + "src/config.py")]
        if not found:
            raise RuntimeError("git clone failed: add a GITHUB_TOKEN secret or attach the project zip as a dataset")
        print("WARNING: git clone failed; using the attached copy at", os.path.dirname(os.path.dirname(found[0])))
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), PROJECT,
                        ignore=shutil.ignore_patterns("data", "outputs", "__pycache__"))

shutil.rmtree("/kaggle/working/project", ignore_errors=True)  # remove stale copies saved by older notebook versions
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip install -q grad-cam anthropic
!git -C {PROJECT} log -1 --format="code version: %h  %s  (%cd)" --date=short 2>/dev/null || echo "code version: attached copy (no git)"

In [ ]:
%matplotlib inline
import os
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from src.kaggle_utils import setup, find_all
setup(data=False)
from src.xai import model_selection

faith = find_all(os.path.join("gradcam", "faithfulness.csv"))
summ = find_all(os.path.join("compare", "summary.csv"))
assert faith, "Attach the committed output of 04_gradcam (it must contain outputs/gradcam/faithfulness.csv)"
if not summ:
    print("WARNING: notebook 03's output not attached; accuracy criteria will be skipped.")
GRADCAM_DIR, SUMMARY_CSV, SAVE_DIR = faith[0].parent, (summ[0] if summ else None), "/kaggle/working/outputs/selection"
print("phase 2 results:", GRADCAM_DIR, "| phase 1 results:", SUMMARY_CSV)


In [ ]:
card, gains, focus, figs = model_selection.report(GRADCAM_DIR, SUMMARY_CSV, display=print, save_dir=SAVE_DIR)
plt.show()
display(Markdown("**Scorecard** (sorted by mean rank)"))
display(card.round(4))
display(Markdown(f"**Best overall by the pre-declared criteria: {card.index[0]}** (mean rank {card['mean rank'].iloc[0]:.2f}). "
                 f"Runner-up: {card.index[1]} ({card['mean rank'].iloc[1]:.2f})."))